# VitroVision — ประเมินความพร้อมนำออกอนุบาล + เก็บข้อมูลการเจริญเติบโตของพืช (SAM 3 zero-shot)

ป้อนภาพขวดโหลเพาะเลี้ยงเนื้อเยื่อ (tissue culture) หลายใบ → ระบบคำนวณ feature เชิงปริมาณ แล้วแสดง
**ตารางเปรียบเทียบระหว่างภาพ** + **สรุปเปรียบเทียบข้ามชนิด** พร้อม verdict ว่า "พร้อมนำออกอนุบาล" รึยัง
แบบ **zero-shot ด้วย SAM 3 (`facebook/sam3`)** — ไม่ต้องเทรนโมเดลใดๆ

## ใช้ได้กับทุกชนิดพืช (ไม่จำกัดชนิด)
- **ชนิดพืชเป็นแค่ label ประกอบ** — ใช้สำหรับตั้ง threshold เฉพาะชนิดและจัดกลุ่มเปรียบเทียบ
  เจ้าของสามารถเพิ่ม/ลบชนิดได้เอง (เช่น เพิ่ม "กล้วย", "ไผ่") ที่ cell CONFIG โดยไม่ต้องแก้โค้ดส่วนอื่น
- auto-match ชนิดจากชื่อไฟล์ (keyword ใน `SPECIES_KEYWORDS`) — ภาพที่ match ไม่ได้ให้เลือกด้วย dropdown
- เลือก **"ไม่ระบุชนิด"** ได้ (ภาพที่ยังไม่รู้ชนิด หรือชนิดใหม่ที่ยังไม่ได้ตั้ง threshold) —
  ภาพกลุ่มนี้จะใช้ **threshold ค่ากลางข้ามชนิด** (COVERAGE_READY/COVERAGE_OVERDENSE) verdict อาจแม่นน้อยกว่า
- CSV ที่ส่งออกมีคอลัมน์ `captured_at` (เวลารันวิเคราะห์) — ใช้ต่อยอดลาก **growth curve** ข้ามรอบการเก็บข้อมูล

## วิธีใช้
1. เลือก **GPU T4**: ไปที่ `Runtime → Change runtime type → T4 GPU` (ต้อง rerun ทั้ง notebook หลังเปลี่ยน)
2. รัน cell ตามลำดับ: install → login → config → load model → upload ภาพ → **ระบุชนิด** (auto-match → dropdown → ยืนยัน) → วิเคราะห์ → overlay → สรุปข้ามชนิด → export
3. แนะนำให้ตั้งชื่อไฟล์ภาพให้มี keyword ชนิดพืช (เช่น `พริก_1.jpg`, `kok_02.png`) เพื่อให้ auto-match ตรง —
   ภาพที่ match ไม่ได้จะให้เลือกชนิดด้วย dropdown ทีละภาพ
4. ดูตาราง verdict + สรุปข้ามชนิด + ดาวน์โหลด CSV

## คำเตือน (⚠️ อ่านก่อน)
- `facebook/sam3` เป็น **gated model** ต้อง login ผ่าน `notebook_login()` ด้วยบัญชีที่ได้รับสิทธิ์แล้ว
  (ขอสิทธิ์ได้ที่ https://huggingface.co/facebook/sam3)
- ⚠️ **ห้ามเปลี่ยนชื่อ model เป็น `facebook/sam3.1`** — API และผลลัพธ์ต่างกัน (ผ่านการทดสอบเฉพาะ `sam3` เท่านั้น)
- ต้องรันบน **GPU runtime** — บน CPU จะช้ามากจนใช้ไม่ได้
- ขวดแก้วโค้งทำให้เกิด **refraction/glare** ซึ่งลดคุณภาพ segmentation — มี `glare_score` คอยลด confidence เป็น safeguard
- **ipywidgets:** ค่าที่เลือกใน dropdown ต้องกดปุ่ม "ยืนยัน" ทุกครั้งหลัง rerun cell นั้น (widget state ไม่คงอยู่ข้าม rerun)

## 🔴 คำเตือนใหญ่: threshold ยังไม่ validate กับแล็บ
ค่า `SPECIES_THRESHOLDS` (ค่าเริ่มต้น ready = 0.35, overdense = 0.80) เป็น **ค่าประมาณคร่าวๆ
จาก literature ข้ามชนิดพืช** (อ้างอิง `research/subculture_criteria.md`) — **ยังไม่ผ่านการเทียบกับข้อมูลจริงในห้องแล็บ**
- จาก literature ค่า threshold น่าจะ**ต่างกันตามชนิดพืช**/สูตรอาหาร/สภาวะเพาะเลี้ยง — ปรับได้เป็นรายชนิดที่ cell CONFIG
- ชนิดที่ไม่อยู่ใน `SPECIES_THRESHOLDS` (รวมถึง "ไม่ระบุชนิด") ใช้ค่า default — verdict เป็นเพียงแนวทางเบื้องต้น
- ผล verdict ใช้เป็น**แนวทางเบื้องต้นเท่านั้น** ห้ามใช้ตัดสินใจจัดการจริงโดยไม่ตรวจกับแล็บก่อน

## Feature ที่คำนวณ (นิยามตรึงจาก `_orchestration.md`)
| feature | นิยาม |
|---|---|
| `coverage_ratio` | area(union ของ mask ทั้งหมด plant+shoot+leaf) / area(ROI) |
| `height_proxy` | bbox_height(union mask plant+shoot) / ROI_height |
| `width_proxy` | bbox_width(union mask plant+shoot) / ROI_width |
| `leaf_count` | จำนวน mask ของ prompt "leaf" (conf ≥ 0.5) |
| `shoot_count` | จำนวน mask ของ "plant" + "shoot" (conf ≥ 0.5) |
| `total_area_px` | พื้นที่รวมของ union mask (plant+shoot+leaf) ภายใน ROI (พิกเซล) |
| `greenness` | ค่าเฉลี่ย G/(R+G+B) เฉพาะ pixel ใน union mask |
| `mean_score` | ค่าเฉลี่ย confidence ของทุก mask |
| `glare_score` (เสริม) | สัดส่วน pixel ใน ROI ที่ V(HSV) > 0.95 และ S < 0.15 — ใช้ลด confidence ไม่ใช้ตัดสินตรงๆ |

ROI = บริเวณขวด (ค่าเริ่มต้น = ทั้งภาพ; เปิด `DETECT_BOTTLE=True` จะใช้ bbox ของ mask ขวดแทน)

หมายเหตุ: ทุกแถวในตารางผลลัพธ์มีคอลัมน์ `captured_at` = เวลารันวิเคราะห์ (ค่าเดียวกันทั้งรอบ) ไว้ลากกราฟ growth ข้ามรอบเก็บข้อมูล

## Verdict (rule-based จาก coverage_ratio ตามชนิดพืช)
- ready ≤ coverage ≤ overdense → **"พร้อมอนุบาล"**
- coverage < ready → **"ยังไม่พร้อม"**
- coverage > overdense → **"หนาแน่นเกิน-ตรวจ"**
- ชนิดที่ไม่อยู่ใน `SPECIES_THRESHOLDS` หรือ "ไม่ระบุชนิด" → ใช้ ready/overdense ค่ากลางข้ามชนิด


In [ ]:
# ติดตั้งไลบรารีที่ต้องใช้ (รันครั้งเดียว)
!pip install -q transformers torch torchvision opencv-python pillow matplotlib pandas numpy huggingface_hub ipywidgets


In [ ]:
# เข้าสู่ระบบ Hugging Face — ต้องใช้บัญชีที่ได้รับสิทธิ์ gated model facebook/sam3
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# ===== CONFIG — ปรับค่าได้ที่นี่โดยไม่ต้องแตะโค้ดอื่น =====

# ---- ชนิดพืชที่วิเคราะห์ (แก้เพิ่ม/ลบได้อิสระ — ไม่จำกัดชนิด) ----
# เจ้าของสามารถเพิ่มชนิดอื่น (เช่น "กล้วย", "ไผ่") หรือลบชนิดที่ไม่ใช้ ได้โดยไม่ต้องแก้โค้ดส่วนอื่น
# ชนิดที่เพิ่มใหม่ ถ้าไม่มี keyword ใน SPECIES_KEYWORDS จะต้องเลือกด้วย dropdown (หรือเลือก "ไม่ระบุชนิด")
SPECIES_LIST = ["พริก", "กก", "ไพล", "ม่วงเทพรัตน์"]

# keyword สำหรับ auto-match ชนิดพืชจากชื่อไฟล์ (เปรียบเทียบเป็นตัวพิมพ์เล็กทั้งหมด ใช้ in บนชื่อไฟล์)
# ชนิดไหนไม่มี keyword (หรือ match ไม่ได้) → เลือกด้วย dropdown ใน cell ถัดไป
SPECIES_KEYWORDS = {
    "พริก": ["พริก", "prick", "chili"],
    "กก": ["กก", "kok", "sedge"],
    "ไพล": ["ไพล", "plai"],
    "ม่วงเทพรัตน์": ["ม่วง", "muang", "tradescantia"],
}

# ---- threshold เฉพาะชนิดพืช ----
# ถ้าชนิดไม่อยู่ใน dict นี้ หรือภาพเป็น "ไม่ระบุชนิด" → ใช้ค่า default COVERAGE_READY/COVERAGE_OVERDENSE (ค่ากลางข้ามชนิด)
# ค่าเริ่มต้นเท่ากันทุกชนิด (ready=0.35, overdense=0.80) — ค่าประมาณจาก literature ข้ามชนิด ยังไม่ validate กับแล็บ
# ตาม literature ค่า threshold ต่างกันตามชนิดพืช/สูตรอาหาร — ปรับเป็นรายชนิดได้ เช่น:
# SPECIES_THRESHOLDS["พริก"] = {"ready": 0.40, "overdense": 0.85}
SPECIES_THRESHOLDS = {sp: {"ready": 0.35, "overdense": 0.80} for sp in SPECIES_LIST}

# ---- SAM3 / segmentation ----
PROMPTS = ["leaf", "plant", "shoot"]  # พรอมต์ของ SAM3 (leaf ใช้นับใบ, plant+shoot ใช้นับยอดและวัดความสูง/กว้าง)
SCORE_THRESHOLD = 0.5                 # เกณฑ์ confidence ต่ำสุดของ mask ใช้ทั้งตอน post-process และนับ leaf_count/shoot_count
DETECT_BOTTLE = False                 # True = ให้ SAM3 segment "glass jar bottle" แล้วใช้ bbox ของ mask ขวดเป็น ROI (กันพื้นหลัง แต่ช้ากว่า)
BASE_CONFIDENCE = 0.8                 # confidence เริ่มต้นของ verdict ก่อนปรับลดด้วย glare_score

# ---- fallback (ใช้เมื่อ species ไม่อยู่ใน SPECIES_THRESHOLDS) ----
COVERAGE_READY = 0.35
COVERAGE_OVERDENSE = 0.80


In [ ]:
import time
import torch
from transformers import Sam3Processor, Sam3Model

# ตรวจ device — ต้องเป็น GPU (Colab T4)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("⚠️ ยังไม่ได้เลือก GPU — ไปที่ Runtime > Change runtime type > T4 GPU แล้ว rerun ตั้งแต่ต้น")

print("กำลังโหลด facebook/sam3 (รอบแรกอาจใช้เวลา 1-2 นาที) ...")
t0 = time.time()
model = Sam3Model.from_pretrained("facebook/sam3").to(device)  # ⚠️ ห้ามเปลี่ยนเป็น facebook/sam3.1
processor = Sam3Processor.from_pretrained("facebook/sam3")
print(f"โหลดเสร็จใน {time.time() - t0:.1f} วินาที")


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab import files

# อัปโหลดภาพขวดโหลเพาะเลี้ยงเนื้อเยื่อ 1 ใบขึ้นไป (jpg/png)
uploaded = files.upload()
names = list(uploaded.keys())
print(f"ได้รับ {len(names)} ภาพ: {names}")

# อ่านเป็น RGB และเก็บใน dict
images = {}
for name in names:
    arr = cv2.imdecode(np.frombuffer(uploaded[name], np.uint8), cv2.IMREAD_COLOR)
    images[name] = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
    print(f"  {name}: {images[name].shape}")

# แสดงภาพทั้งหมด
fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
if len(images) == 1:
    axes = [axes]
for ax, name in zip(axes, names):
    ax.imshow(images[name])
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.show()


In [ ]:
# ===== ระบุชนิดพืชอัตโนมัติจากชื่อไฟล์ =====
# ใช้ keyword (ตัวพิมพ์เล็ก) หาในชื่อไฟล์ — ภาพที่ match ไม่ได้ ต้องเลือกชนิดเองใน cell ถัดไป

def match_species(filename):
    fname = filename.lower()
    for sp in SPECIES_LIST:
        for kw in SPECIES_KEYWORDS[sp]:
            if kw in fname:
                return sp
    return None

auto_species = {}
unmatched = []
for name in names:
    sp = match_species(name)
    if sp:
        auto_species[name] = sp
    else:
        unmatched.append(name)

print(f"auto-match ได้ {len(auto_species)}/{len(names)} ภาพ")
for name in names:
    sp = auto_species.get(name)
    status = sp if sp else "❌ ยังไม่ระบุ (เลือกใน cell ถัดไป)"
    print(f"  {name} → {status}")


In [ ]:
# ===== เลือกชนิดพืชด้วย dropdown สำหรับภาพที่ auto-match ไม่ได้ =====
# เลือกชนิดให้ภาพที่แสดง (หรือเลือก "ไม่ระบุชนิด") แล้วกดปุ่ม "ยืนยัน species"
# — กดยืนยันได้เสมอ ภาพที่ยังไม่เลือก/เลือก "ไม่ระบุชนิด" จะถือเป็น "ไม่ระบุชนิด" โดยปริยาย
# ⚠️ ทุกครั้งที่ rerun cell นี้ต้องกดยืนยันใหม่ (widget state ไม่คงอยู่ข้าม rerun)

import ipywidgets as widgets
from IPython.display import display, Image as IPImage

UNSPECIFIED_LABEL = "ไม่ระบุชนิด"
manual_species = {}

if unmatched:
    print(f"เลือกชนิดพืชให้ {len(unmatched)} ภาพ (เลือก 'ไม่ระบุชนิด' ได้) แล้วกดปุ่มยืนยัน:")
    widgets_manual = {}
    for name in unmatched:
        display(IPImage(data=uploaded[name], width=120))  # แสดงรูปเล็กของภาพนั้น
        w = widgets.Dropdown(options=[UNSPECIFIED_LABEL] + SPECIES_LIST, description=name, value=None,
                             layout=widgets.Layout(width="380px"))
        widgets_manual[name] = w
        display(w)

    btn = widgets.Button(description="ยืนยัน species")
    out = widgets.Output()

    def on_confirm(b):
        with out:
            out.clear_output()
            manual_species.clear()
            chosen = []
            for n, w in widgets_manual.items():
                if w.value is not None and w.value != UNSPECIFIED_LABEL:
                    manual_species[n] = w.value
                    chosen.append(n)
            unspecified = [n for n in widgets_manual if n not in manual_species]
            print(f"บันทึกแล้ว: ระบุชนิด {len(chosen)} ภาพ | ไม่ระบุชนิด {len(unspecified)} ภาพ")
            if unspecified:
                print("  ไม่ระบุชนิด: " + ", ".join(unspecified))

    btn.on_click(on_confirm)
    display(btn, out)
else:
    print("ทุกภาพ auto-match ได้ครบ ไม่ต้องเลือก manual")


In [ ]:
# ===== รวม species ทั้งหมด (ต้องรัน cell นี้: auto-match + manual) =====
# ภาพที่ไม่มี species (auto-match ไม่ได้ + ไม่ได้เลือกใน dropdown) → กำหนดเป็น "ไม่ระบุชนิด"
# ไม่ hard-block — "ไม่ระบุชนิด" ใช้ threshold default (ค่ากลางข้ามชนิด)

import pandas as pd

UNSPECIFIED = "ไม่ระบุชนิด"

species_map = {}
for n in names:
    sp = auto_species.get(n) or manual_species.get(n)
    species_map[n] = sp if sp else UNSPECIFIED

unspecified_n = [n for n in names if species_map[n] == UNSPECIFIED]
if unspecified_n:
    print(f"⚠️ มี {len(unspecified_n)}/{len(names)} ภาพเป็น 'ไม่ระบุชนิด': {unspecified_n}")
    print("   ถ้ารู้ชนิด ควรกลับไประบุใน cell dropdown — verdict จะใช้ threshold เฉพาะชนิด (แม่นขึ้น)")
else:
    print(f"ครบทุกภาพ ({len(species_map)}) — ทุกภาพระบุชนิดแล้ว")

df_species = pd.DataFrame([{"image": n, "species": species_map[n]} for n in names])
df_species


In [ ]:
# ===== ฟังก์ชันคำนวณ feature (numpy/cv2 ล้วน ไม่มี dependency เพิ่ม) =====

def masks_to_numpy(result):
    """แปลง masks (N×H×W) และ scores จากผล SAM3 เป็น numpy"""
    masks = result["masks"]
    if hasattr(masks, "cpu"):
        masks = masks.cpu()
    masks = np.asarray(masks).astype(bool)
    scores = result.get("scores")
    if scores is not None:
        if hasattr(scores, "cpu"):
            scores = scores.cpu()
        scores = np.asarray(scores)
    return masks, scores

def union_mask(masks):
    """รวม mask หลายตัวเป็น mask เดียว (OR) — คืน None ถ้าไม่มี mask"""
    if len(masks) == 0:
        return None
    return masks.any(axis=0)

def union_bbox(mask):
    """bbox (x, y, w, h) ของ mask — คืน None ถ้า mask ว่าง"""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return xs.min(), ys.min(), xs.max() - xs.min() + 1, ys.max() - ys.min() + 1

def segment_prompt(img_rgb, prompt):
    """รัน SAM3 แบบ text-prompt บน 1 ภาพ คืน (masks, scores)"""
    inputs = processor(images=img_rgb, text=prompt, return_tensors="pt").to(device)
    outputs = model(**inputs)
    result = processor.post_process_instance_segmentation(
        outputs, threshold=SCORE_THRESHOLD, mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist()
    )[0]
    return masks_to_numpy(result)

def count_confident(scores, masks):
    """นับ mask ที่ confidence >= SCORE_THRESHOLD"""
    if scores is not None:
        return int((scores >= SCORE_THRESHOLD).sum())
    return len(masks)

def analyze_image(img_rgb, species):
    """วิเคราะห์ 1 ภาพ → dict ของ feature + verdict + confidence + note + masks ต่อ prompt
    threshold verdict: ใช้ค่าเฉพาะชนิดจาก SPECIES_THRESHOLDS
    — ชนิดที่ไม่อยู่ใน dict (รวมถึง "ไม่ระบุชนิด") fallback เป็นค่า default COVERAGE_READY/COVERAGE_OVERDENSE (ค่ากลางข้ามชนิด)"""
    H, W = img_rgb.shape[:2]
    th = SPECIES_THRESHOLDS.get(species, {"ready": COVERAGE_READY, "overdense": COVERAGE_OVERDENSE})

    # 1) segment ทุก prompt ที่กำหนดใน config
    masks_by_prompt = {}
    for prompt in PROMPTS:
        masks, scores = segment_prompt(img_rgb, prompt)
        masks_by_prompt[prompt] = (masks, scores)

    # 2) ROI — ค่าเริ่มต้น = ทั้งภาพ, ถ้า DETECT_BOTTLE=True ใช้ bbox ของ mask "glass jar bottle"
    roi = np.ones((H, W), dtype=bool)
    roi_h, roi_w = H, W
    if DETECT_BOTTLE:
        b_masks, _ = segment_prompt(img_rgb, "glass jar bottle")
        if len(b_masks) > 0:
            bb = union_bbox(union_mask(b_masks))
            if bb is not None:
                x, y, w, h = bb
                roi = np.zeros((H, W), dtype=bool)
                roi[y:y + h, x:x + w] = True
                roi_h, roi_w = h, w
        else:
            print("  ⚠️ ไม่พบ mask ขวด ใช้ทั้งภาพเป็น ROI แทน")
    roi_area = int(roi.sum())

    # 3) coverage_ratio + total_area_px — union ของ mask ทั้งหมด plant+shoot+leaf (ภายใน ROI)
    cov_masks = np.concatenate([masks_by_prompt[p][0] for p in PROMPTS])
    union_cov = cov_masks.any(axis=0)
    mask_in_roi = union_cov & roi
    cov_area = int(mask_in_roi.sum())
    coverage = cov_area / max(roi_area, 1)
    total_area_px = cov_area

    # 4) height_proxy + width_proxy — bbox ของ union mask plant+shoot เทียบกับ ROI
    ph_prompts = [p for p in ["plant", "shoot"] if p in masks_by_prompt]
    if not ph_prompts:  # เผื่อกรณีปรับ PROMPTS เหลือ 1 ตัว
        ph_prompts = list(masks_by_prompt.keys())
    ph_masks = np.concatenate([masks_by_prompt[p][0] for p in ph_prompts])
    bb_ph = union_bbox(ph_masks.any(axis=0))
    if bb_ph is not None:
        height_proxy = bb_ph[3] / max(roi_h, 1)
        width_proxy = bb_ph[2] / max(roi_w, 1)
    else:
        height_proxy, width_proxy = 0.0, 0.0

    # 5) leaf_count = จำนวน mask ของ "leaf" (conf >= 0.5)
    lm, ls = masks_by_prompt.get("leaf", (np.zeros((0, H, W), dtype=bool), None))
    leaf_count = count_confident(ls, lm)

    #    shoot_count = จำนวน mask ของ "plant" + "shoot" (conf >= 0.5)
    shoot_count = 0
    for p in ["plant", "shoot"]:
        m, s = masks_by_prompt.get(p, (np.zeros((0, H, W), dtype=bool), None))
        shoot_count += count_confident(s, m)

    # 6) greenness = ค่าเฉลี่ย G/(R+G+B) เฉพาะ pixel ใน union mask (ภายใน ROI)
    rgb_f = img_rgb.astype(np.float32)
    g_ratio = rgb_f[:, :, 1] / (rgb_f.sum(axis=2) + 1e-6)
    greenness = float(g_ratio[mask_in_roi].mean()) if mask_in_roi.any() else 0.0

    # 7) mean_score = ค่าเฉลี่ย confidence ของทุก mask ทุก prompt
    score_list = [s for _, s in masks_by_prompt.values() if s is not None and len(s) > 0]
    mean_score = float(np.concatenate(score_list).mean()) if score_list else 0.0

    # 8) glare_score = สัดส่วน pixel ใน ROI ที่ V(HSV) > 0.95 และ S < 0.15 (safeguard ลด confidence)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    v = hsv[:, :, 2].astype(np.float32) / 255.0
    s = hsv[:, :, 1].astype(np.float32) / 255.0
    glare_score = float(((v > 0.95) & (s < 0.15) & roi).sum()) / max(roi_area, 1)

    # 9) verdict — rule-based จาก coverage ตาม threshold เฉพาะชนิด (ยังไม่ validate กับแล็บ)
    if coverage >= th["overdense"]:
        verdict = "หนาแน่นเกิน-ตรวจ"
    elif coverage >= th["ready"]:
        verdict = "พร้อมอนุบาล"
    else:
        verdict = "ยังไม่พร้อม"

    # 10) confidence — base คูณลดตาม glare_score
    confidence = max(BASE_CONFIDENCE * (1.0 - glare_score), 0.0)

    # 11) note — เตือนสัญญาณผสม
    notes = []
    if coverage >= th["ready"] and height_proxy < 0.3:
        notes.append("coverage สูงแต่ height_proxy ต่ำผิดปกติ (อาจติด glare/พื้นหลัง)")
    if glare_score > 0.4:
        notes.append("glare สูง confidence ถูกลดแล้ว")
    if leaf_count == 0 and coverage > 0.05:
        notes.append("มี coverage แต่ไม่พบ mask ใบ")
    note = " | ".join(notes)

    return {
        "coverage_ratio": coverage,
        "height_proxy": height_proxy,
        "width_proxy": width_proxy,
        "leaf_count": leaf_count,
        "shoot_count": shoot_count,
        "total_area_px": total_area_px,
        "greenness": greenness,
        "mean_score": mean_score,
        "glare_score": glare_score,
        "verdict": verdict,
        "confidence": confidence,
        "note": note,
        "masks_by_prompt": masks_by_prompt,
    }


In [ ]:
import pandas as pd
from datetime import datetime

# เวลารันวิเคราะห์ (ค่าเดียวกันทั้งชุดในรอบนี้) — ไว้ลากกราฟการเจริญเติบโต (growth curve) ข้ามรอบการเก็บข้อมูล
captured_at = datetime.now().strftime("%Y-%m-%d %H:%M")

results_all = {}
rows = []
for i, name in enumerate(names):
    sp = species_map[name]
    print(f"[{i+1}/{len(names)}] กำลังวิเคราะห์: {name} ({sp})")
    res = analyze_image(images[name], sp)
    results_all[name] = res
    rows.append({
        "image": name,
        "species": sp,
        "captured_at": captured_at,
        "verdict": res["verdict"],
        "confidence": round(res["confidence"], 3),
        "coverage_ratio": round(res["coverage_ratio"], 4),
        "height_proxy": round(res["height_proxy"], 4),
        "width_proxy": round(res["width_proxy"], 4),
        "leaf_count": res["leaf_count"],
        "shoot_count": res["shoot_count"],
        "total_area_px": res["total_area_px"],
        "greenness": round(res["greenness"], 4),
        "mean_score": round(res["mean_score"], 4),
        "glare_score": round(res["glare_score"], 4),
        "note": res["note"],
    })

df = pd.DataFrame(rows, columns=[
    "image", "species", "captured_at", "verdict", "confidence", "coverage_ratio", "height_proxy",
    "width_proxy", "leaf_count", "shoot_count", "total_area_px", "greenness",
    "mean_score", "glare_score", "note",
])

# จัดเรียง: ตามชนิดพืช (ลำดับ SPECIES_LIST) แล้วตาม verdict (ความพร้อม) แล้วตาม coverage
# — "ไม่ระบุชนิด" ไม่อยู่ในลำดับ SPECIES_LIST → ถูกจัดไว้ท้ายสุด (NaN sort)
sp_order = {sp: i for i, sp in enumerate(SPECIES_LIST)}
verdict_order = {"พร้อมอนุบาล": 0, "หนาแน่นเกิน-ตรวจ": 1, "ยังไม่พร้อม": 2}
df["_sp"] = df["species"].map(sp_order)
df["_v"] = df["verdict"].map(verdict_order)
df = df.sort_values(["_sp", "_v", "coverage_ratio"], ascending=[True, True, False]).drop(columns=["_sp", "_v"]).reset_index(drop=True)

print(f"วิเคราะห์ครบ {len(df)} ภาพ (captured_at: {captured_at})")
df


In [ ]:
# ===== overlay contour สีเขียว แยกต่อ prompt (1 แถว 3 คอลัมน์: leaf/plant/shoot) =====
for name in names:
    res = results_all[name]
    img = images[name].copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, prompt in zip(axes, PROMPTS):
        masks, scores = res["masks_by_prompt"][prompt]
        overlay = img.copy()
        for mask in masks:
            m = mask.astype(np.uint8)
            contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)
        n = count_confident(scores, masks)
        ax.imshow(overlay)
        ax.set_title(f"{prompt}: {n} mask")
        ax.axis("off")
    fig.suptitle(
        f"{name} | {species_map[name]} | verdict: {res['verdict']} | confidence: {res['confidence']:.2f} | "
        f"coverage: {res['coverage_ratio']:.3f} | height: {res['height_proxy']:.3f} | width: {res['width_proxy']:.3f} | "
        f"leaf: {res['leaf_count']} | shoot: {res['shoot_count']} | glare: {res['glare_score']:.3f}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()


In [ ]:
# ===== สรุปเปรียบเทียบข้ามชนิด (rows = ชนิดพืช, columns = เฉลี่ย/เบี่ยงเบนของ feature + verdict) =====

feat_cols = ["coverage_ratio", "height_proxy", "width_proxy", "leaf_count", "shoot_count",
             "total_area_px", "greenness", "mean_score", "glare_score"]

# ค่าเฉลี่ย + ค่าเบี่ยงเบนมาตรฐาน ของแต่ละ feature ต่อชนิด
summ = df.groupby("species")[feat_cols].agg(["mean", "std"]).round(4)
print("=== สรุปเฉลี่ย feature ต่อชนิดพืช (mean / std) ===")
summ

# จำนวนภาพต่อ verdict ต่อชนิด — เทียบว่าชนิดไหนพร้อมอนุบาลมากกว่า
verdict_tab = df.groupby("species")["verdict"].value_counts().unstack(fill_value=0)
verdict_tab = verdict_tab.reindex(columns=["พร้อมอนุบาล", "หนาแน่นเกิน-ตรวจ", "ยังไม่พร้อม"], fill_value=0)
verdict_tab["รวมภาพ"] = verdict_tab.sum(axis=1)
print("\n=== จำนวนภาพต่อ verdict ต่อชนิด ===")
verdict_tab


In [ ]:
# ===== export CSV (utf-8-sig เปิดกับ Excel ได้): ผลรายภาพ + สรุปข้ามชนิด =====
# CSV หลักมีคอลัมน์ species + captured_at — ใช้ต่อยอดวิเคราะห์การเจริญเติบโต (growth curve) ข้ามรอบเก็บข้อมูล
csv_name = "sam3_readiness_results.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
files.download(csv_name)
print(f"ดาวน์โหลด {csv_name} แล้ว")

summ_csv = "sam3_readiness_summary.csv"
summ.to_csv(summ_csv, index=True, encoding="utf-8-sig")
files.download(summ_csv)
print(f"ดาวน์โหลด {summ_csv} แล้ว")

verdict_csv = "sam3_readiness_verdict_counts.csv"
verdict_tab.to_csv(verdict_csv, index=True, encoding="utf-8-sig")
files.download(verdict_csv)
print(f"ดาวน์โหลด {verdict_csv} แล้ว")
